<!-- @@FENCINGLY_DURACION_COMPLETA:protocolo -->
# 00 - REGLA DE PROTOCOLO. Leer antes de ejecutar y antes de interpretar

**El numero que salga de esta corrida es una LINEA BASE NUEVA. No se puede comparar con el 73,88 %,
ni con el 70,04 %, ni con ninguna cifra anterior del proyecto.**

Con duracion completa cambian `pre_s_eff`, `n_frames_window`, cada rasgo derivado de la ventana y la
huella del productor. Es otra cadena de medida. Restar una cifra de un protocolo a una cifra de otro
protocolo es un **error de protocolo**, no una comparacion conservadora. El informe del proyecto ya
lleva esa disciplina escrita ("nowhere in this report is a figure from one protocol subtracted from a
figure of another") y aqui aplica igual.

Lo que **si** se puede comparar: duracion-completa contra duracion-completa, con y sin el cambio que
se este probando, **pareado por clip**.

## El veredicto se reporta en dos poblaciones, siempre

El atajo "gana el que enciende segundo" acierta **63,51 %** por si solo, y el modelo lo predice en el
**72,71 %** de los clips: lo sobreusa. Eso parte el corpus:

| poblacion | n | acierto |
|---|---:|---:|
| gano el que encendio **segundo** | 1.671 | **83,00 %** |
| gano el que encendio **primero** | 960 | **45,21 %** |
| brecha | | **+37,80 pp** |

* llevar los 960 de 45,21 % a **60 %** vale **+5,40 pp** globales; a **75 %**, **+10,95 pp**
* degradar 5 pp los 1.671 cuesta **-3,18 pp**

**El veredicto es la media global pareada.** El subgrupo describe donde cayo la ganancia; nunca es la
poblacion de evaluacion, nunca un objetivo de optimizacion y nunca una puerta de enrutado. Evaluar
dentro de un subgrupo definido por el resultado hace que la base caiga bajo el azar: alli un bloque
sin informacion compro **+3,08 pp**.

## Limitacion honesta, y va en el informe

Con la potencia de este corpus **un efecto real de +1,6 pp es indetectable**, y hacen falta **12 a 20
campeonatos mas** para tener el 80 % de potencia sobre ese tamano de efecto. **Una mejora real puede
no ser demostrable aunque exista.** Este cuaderno no promete lo que no puede medir.

## Varas de medir, para no creerse un delta

* Cambiar **solo** el backend de atencion, misma receta, misma semilla y mismo interprete, da
  **+0,36 pp con 327 de 3.047 decisiones distintas**. Cualquier mejora de red por debajo de eso es
  indistinguible de recompilar. (Por eso la celda de preflight fuerza el backend `math`.)
* El nulo se elige **por el tipo de rasgo**. El desplazamiento circular **preserva** los rasgos de
  nivel (una columna quedo identica al 100 %), **infla** los de derivada (hasta 7,79x) y es **inerte**
  sobre representaciones agrupadas (coseno 0,9980). Para nivel, nulo de **signo de lado aleatorio**.
* **Dos convenios de promediado de semillas no son la misma cantidad.** El banco de ablaciones
  promedia el *acierto por clip*; el artefacto promedia las *probabilidades* y umbraliza una vez. Un
  efecto de +1,03 con el primero fue **+0,80 con intervalo cruzando el cero** con el segundo. **Usa
  el del artefacto.**


# 02 · Entrenamiento pequeño y evaluación trazable — segundo paso efectivo

**Entrada:** workspace válido de 01 y código del mismo release. **Salida fundamental:** workspace completo, ahora con
`model_bundle.zip` en la raíz, más selección, historia y evaluación. Importa ese ZIP en la web local: no necesitas ejecutar
el notebook 03 para usar la aplicación de escritorio/navegador.

La receta envuelve al trainer nativo, no copia un entrenamiento dentro de las celdas. Usa el **mismo TemporalNet** que
recarga la web: proyección → TCN dilatada → Transformer pequeño → pooling con máscara → dos clases left/right.
En el contrato actual, d=64/F=78 tiene **188.063 parámetros**; es una referencia de tamaño, no «el mejor modelo» demostrado.
No se inventa una clase distinta `SharedTemporalNet` ni se cambia su nombre de arquitectura al serializar. «Compartido»
significa mismo código, features, masks y loader entre Colab y React/PC.

Por defecto `RUN_TRAINING=False`, Optuna tiene cero trials y no se ejecuta un modelo grande. La prueba local de entrega
es una integración sintética pequeña en CPU; no convierte seis clips de una demo en evidencia deportiva suficiente.


## Abrir el mismo release privado: ZIP subido, carpeta ya extraída o clon fijado por commit

Necesitas dos cosas diferentes: **el ZIP de la entrega FENCINGLY** (código de confianza) y, a partir del segundo notebook,
**el ZIP de workspace** (sólo datos/modelo). No son intercambiables. No subas un workspace en el lugar del release.
El código de esta entrega no tiene por qué existir en ningún repositorio remoto. No hay una descarga de GitHub por defecto.

La entrega recomendada tiene `FENCINGLY/webapp/fencingly_v2/` y `FENCINGLY/webapp/webapp_react/frontend/dist/`.
Los notebooks viven en `FENCINGLY/colab/notebooks/` y el estudio en `FENCINGLY/study/`. Después del bootstrap, **ROOT apunta
al contenido de `webapp/`**, no a su carpeta padre ni al workspace de datos.

**En Colab:** abre este `.ipynb` con Archivo → Subir notebook. Si quieres acelerar la extracción/entrenamiento, selecciona
Entorno de ejecución → Cambiar tipo de entorno → GPU. Colab decide qué GPU hay; una T4 no está garantizada. Sube el ZIP del
release cuando se solicite, o escribe su ruta en `RELEASE_ZIP` si está en Drive ya montado. Para un fichero grande, monta
Drive primero en una celda propia con `google.colab.drive.mount('/content/drive')` y usa su ruta, evitando cargarlo entero
con el navegador. Si conservas el código extraído en este mismo runtime, proporciona `RELEASE_ROOT` y no vuelvas a extraer.

**En un PC/Jupyter:** descomprime la entrega de confianza y fija `RELEASE_ROOT` a su carpeta `webapp` (por ejemplo
`C:/mis_datos/FENCINGLY/webapp`). No uses una ruta de esta auditoría en otro ordenador. Todos los paths posteriores se
calculan desde ese ROOT y desde tus carpetas explícitas.

La celda mínima de bootstrap sólo lee del ZIP un helper **cuyo SHA-256 está fijado en este notebook**. No ejecuta un helper
arbitrario procedente de un workspace. Ese helper valida las rutas de todo el release, tamaños, expansión, colisiones,
enlaces y presencia del servidor/React `dist` antes de extraer a una carpeta nueva. No usa `extractall`. El estudio y los
notebooks del ZIP completo no se duplican dentro de `webapp/`. Si tienes el SHA-256 publicado **junto a tu entrega privada**,
fíjalo en `EXPECTED_RELEASE_SHA256`: integridad sin una referencia de confianza no equivale a autenticidad del autor.
No cambies el hash del helper para hacer pasar un ZIP desconocido. Si actualizas el código, pide el notebook del mismo release.

### Tercera vía opcional: clonar un repositorio con el commit fijado

Si prefieres no volver a subir el ZIP en cada sesión, fija `FENCINGLY_RELEASE_GIT` a la URL HTTPS del repositorio,
`FENCINGLY_RELEASE_GIT_REF` a su rama o etiqueta y `FENCINGLY_RELEASE_COMMIT` al commit completo de 40 hex que hayas
revisado. Por defecto esas tres variables están vacías y **no se clona nada**: la vía por defecto sigue siendo el ZIP
privado subido a mano, y el flujo sin red externa no cambia. Publicar el código sigue siendo una decisión tuya, no un
requisito de estos notebooks.

El clon es superficial (`--depth 1`) sobre una carpeta nueva y **no relaja la integridad**. Una rama es móvil, así que
el pin de commit es **obligatorio** al clonar: si el `HEAD` clonado no es exactamente ese commit, la celda se detiene.
Después se comprueba además el SHA-256 del helper `fencingly_v2/artifact_bundle.py` contra `HELPER_SHA256`, igual que
en la vía del ZIP. Un commit fijado prueba qué árbol se ejecuta, no quién lo escribió: revisa el repositorio antes de
fijarlo y no actualices el pin sin leer el diff. No se hace `push`, no se publican datos y no se sube nada al clonar.

El repositorio debe exponer el `webapp` en **una sola** de estas tres formas, las mismas que acepta el ZIP:
`FENCINGLY/webapp/`, `webapp/` o la propia raíz. Dentro tienen que estar `fencingly_v2/`, `webapp_react/server.py` y
`webapp_react/frontend/dist/index.html` ya compilado; sin ese `dist` la celda se detiene y no se arranca otro frontend.

La procedencia elegida queda en `RELEASE_PROVENANCE` (origen, URL, ref, commit, hash del helper y si el pin coincidió),
se imprime y viaja con el workspace exportado.


In [ ]:
import os, sys, hashlib, shutil, subprocess, zipfile, types
from pathlib import Path
from datetime import datetime, timezone

RELEASE_ROOT = os.environ.get("FENCINGLY_RELEASE_ROOT", "")
RELEASE_ZIP = os.environ.get("FENCINGLY_RELEASE_ZIP", "")
RELEASE_GIT_URL = os.environ.get("FENCINGLY_RELEASE_GIT", "")
RELEASE_GIT_REF = os.environ.get("FENCINGLY_RELEASE_GIT_REF", "main")
EXPECTED_RELEASE_COMMIT = os.environ.get("FENCINGLY_RELEASE_COMMIT", "")
EXPECTED_RELEASE_SHA256 = os.environ.get("FENCINGLY_RELEASE_SHA256", "")
IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules)
HELPER_SHA256 = "ee0e2a0c5cebf80c41c9b5e15c43c48801173780682553e7f24bd9bbb58848e3"
RELEASE_LAYOUTS = ("FENCINGLY/webapp", "webapp", "")

if RELEASE_ROOT:
    ROOT = Path(RELEASE_ROOT).expanduser().resolve()
    RELEASE_PROVENANCE = {"source": "release_root", "location": str(ROOT), "helper_sha256": HELPER_SHA256,
                          "authenticity": "user_must_trust_release_source"}
elif RELEASE_GIT_URL:
    if not RELEASE_GIT_URL.startswith("https://") or any(character.isspace() for character in RELEASE_GIT_URL):
        raise ValueError("Sólo se clona un remoto HTTPS explícito; no se aceptan otros transportes")
    if not RELEASE_GIT_REF or RELEASE_GIT_REF.startswith("-") or any(character.isspace() for character in RELEASE_GIT_REF):
        raise ValueError("Indica una rama o etiqueta explícita en FENCINGLY_RELEASE_GIT_REF")
    if len(EXPECTED_RELEASE_COMMIT) != 40 or any(character not in "0123456789abcdef" for character in EXPECTED_RELEASE_COMMIT.lower()):
        raise ValueError("Clonar exige FENCINGLY_RELEASE_COMMIT con el commit completo de 40 hex que hayas revisado: una rama es móvil y no acredita integridad")
    if shutil.which("git") is None:
        raise ValueError("Este runtime no trae git; usa RELEASE_ZIP o RELEASE_ROOT")
    parent = Path("/content" if IN_COLAB else Path.cwd())
    clone_root = parent / ("fencingly_clone_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))
    if clone_root.exists():
        raise ValueError("El destino del clon ya existe; no se reutiliza ni se sobrescribe una carpeta previa")
    cloned = subprocess.run(["git", "clone", "--depth", "1", "--branch", RELEASE_GIT_REF, "--", RELEASE_GIT_URL, str(clone_root)],
                            capture_output=True, text=True)
    if cloned.returncode != 0 or not (clone_root / ".git").is_dir():
        raise ValueError("El clon superficial ha fallado: " + " ".join((cloned.stderr or "").split())[:300])
    resolved = subprocess.run(["git", "-C", str(clone_root), "rev-parse", "HEAD"], capture_output=True, text=True)
    cloned_commit = (resolved.stdout or "").strip().lower()
    if resolved.returncode != 0 or len(cloned_commit) != 40:
        raise ValueError("No se ha podido leer el commit clonado; no se ejecuta un árbol sin identificar")
    if cloned_commit != EXPECTED_RELEASE_COMMIT.lower():
        raise ValueError("El commit clonado no coincide con FENCINGLY_RELEASE_COMMIT: revisa el cambio y actualiza el pin; no se ejecuta un árbol no fijado")
    layouts = [clone_root.joinpath(*[part for part in layout.split("/") if part]) for layout in RELEASE_LAYOUTS]
    layouts = [candidate for candidate in layouts if (candidate / "fencingly_v2" / "artifact_bundle.py").is_file()]
    if len(layouts) != 1:
        raise ValueError("El repositorio debe exponer exactamente un webapp con fencingly_v2/artifact_bundle.py en FENCINGLY/webapp, en webapp o en la raíz")
    ROOT = layouts[0].resolve()
    cloned_helper = (ROOT / "fencingly_v2" / "artifact_bundle.py").read_bytes()
    if hashlib.sha256(cloned_helper).hexdigest() != HELPER_SHA256:
        raise ValueError("El helper clonado no coincide con este notebook: usa el commit del mismo release; no omitas el control")
    RELEASE_PROVENANCE = {"source": "git_clone", "url": RELEASE_GIT_URL, "ref": RELEASE_GIT_REF, "commit": cloned_commit,
                          "helper_sha256": HELPER_SHA256, "authenticity": "pinned_commit_matched"}
    print("Release clonado y fijado:", RELEASE_PROVENANCE)
else:
    if not RELEASE_ZIP:
        if not IN_COLAB:
            raise ValueError("Fija RELEASE_ROOT al webapp del release, RELEASE_ZIP al ZIP privado o RELEASE_GIT al repositorio con su commit fijado")
        from google.colab import files
        uploaded_release = files.upload()
        candidates = [name for name in uploaded_release if name.lower().endswith(".zip")]
        if len(candidates) != 1:
            raise ValueError("Sube exactamente un ZIP de RELEASE, no el workspace")
        RELEASE_ZIP = candidates[0]
        del uploaded_release
    if Path(RELEASE_ZIP).stat().st_size > 4 * 1024 ** 3:
        raise ValueError("Release mayor que el presupuesto de 4 GiB")
    with Path(RELEASE_ZIP).open("rb") as release_stream:
        footer = zipfile._EndRecData(release_stream)
        if footer is None or footer[zipfile._ECD_ENTRIES_TOTAL] > 30000 or footer[zipfile._ECD_SIZE] > 8 * 1024 ** 2:
            raise ValueError("Metadatos del ZIP de release excesivos o inválidos antes de cargar su directorio")
    with zipfile.ZipFile(RELEASE_ZIP) as archive:
        if len(archive.infolist()) > 30000:
            raise ValueError("Release con demasiadas entradas")
        approved_names = {prefix + "fencingly_v2/artifact_bundle.py" for prefix in ("FENCINGLY/webapp/", "webapp/", "")}
        helpers = [entry for entry in archive.infolist() if entry.filename in approved_names]
        if len(helpers) != 1 or helpers[0].file_size > 150000 or helpers[0].flag_bits & 1:
            raise ValueError("Helper del release ausente, ambiguo o excesivo")
        helper_bytes = archive.read(helpers[0])
    if hashlib.sha256(helper_bytes).hexdigest() != HELPER_SHA256:
        raise ValueError("El helper no coincide con este notebook: usa el release correspondiente; no omitas el control")
    bootstrap = types.ModuleType("_fencingly_verified_release_bootstrap")
    sys.modules[bootstrap.__name__] = bootstrap
    exec(compile(helper_bytes, "verified_artifact_bundle.py", "exec"), bootstrap.__dict__)
    parent = Path("/content" if IN_COLAB else Path.cwd())
    destination = parent / ("fencingly_release_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))
    extracted = bootstrap.extract_release_to_new_directory(RELEASE_ZIP, destination,
                    expected_sha256=EXPECTED_RELEASE_SHA256 or None)
    ROOT = Path(extracted["root"])
    RELEASE_PROVENANCE = {"source": "release_zip", "sha256": extracted["sha256"], "webapp_prefix": extracted["webapp_prefix"],
                          "helper_sha256": HELPER_SHA256, "authenticity": extracted["authenticity"]}
    print("Release validado:", extracted)

required = ("fencingly_v2/colab_workflow.py", "fencingly_v2/training_recipe.py", "fencingly_v2/epoch_checkpoint.py",
            "fencingly_v2/measurement_artifacts.py", "webapp_react/server.py", "webapp_react/frontend/dist/index.html")
if not all((ROOT / relative).is_file() for relative in required):
    raise ValueError("ROOT no contiene el webapp completo del mismo release, incluido React dist")
if "fencingly_v2" in sys.modules and Path(sys.modules["fencingly_v2"].__file__).resolve().parent != ROOT / "fencingly_v2":
    raise RuntimeError("Otro release ya está importado: reinicia el kernel; no mezcles bindings de fuentes distintas")
sys.path.insert(0, str(ROOT))
os.environ["FENCINGLY_RELEASE_ROOT"] = str(ROOT)
from fencingly_v2 import colab_workflow as CW, artifact_bundle as AB
print("ROOT del código compartido:", ROOT)


## Dependencias: comprobar antes de instalar

La entrega no presupone que Colab traiga las versiones correctas. El informe muestra Python y módulos realmente
instalados. La instalación está **desactivada**. Si falta algo, activa `INSTALL_MISSING=True` de forma consciente;
el helper sólo solicita los módulos de esa fase. Nunca instala automáticamente torch/CUDA cuando torch falta: elige
el runtime y la distribución oficial adecuados, reinicia si el instalador lo pide y repite la comprobación. No copies un
entorno virtual de Windows a Colab. No se ha instalado aquí un stack GPU para aparentar una prueba en CUDA.

Un import correcto no verifica la compatibilidad de un Store o modelo: esos artefactos conservan sus propios contratos.
La parte web puede necesitar dependencias opcionales adicionales para informes/PDF; usa las requirements de la misma
entrega y los mensajes del servidor, no una lista de paquetes de otro notebook histórico.


In [ ]:
INSTALL_MISSING = False
N_TRIALS = int(os.environ.get("FENCINGLY_TRIALS", "0"))
TRAIN_DEPENDENCIES = CW.dependency_report("train", include_search=N_TRIALS > 0)
print(TRAIN_DEPENDENCIES)
if INSTALL_MISSING:
    TRAIN_DEPENDENCIES = CW.install_missing("train", approved=True, include_search=N_TRIALS > 0)
from fencingly_v2 import training_recipe as RECIPE
if not hasattr(RECIPE, "SignalTrainingRecipe"):
    raise RuntimeError("Este notebook requiere el release con evaluación de señales; usa el notebook y el código de la misma entrega")
print("Contrato de exportación:", RECIPE.portability_capability())

## Dónde viven los datos y cómo sobrevivir a una desconexión

`WORK` es una carpeta **nueva de trabajo**, distinta de ROOT y de tus originales. En Colab se procesa sobre disco local:
Drive es lento con miles de ficheros pequeños. Se copian snapshots ZIP verificados a Drive y el registro GLOBAL se mantiene
allí directamente. En local puedes elegir un disco propio. No reutilices la carpeta histórica de otra versión como destino.

`USE_DRIVE=True` en Colab pide tu autorización para montar Drive. Si no quieres hacerlo, fija False y descarga el ZIP al
terminar cada fase; perder el runtime antes de esa descarga pierde lo que sólo estuviese en `/content`. Un path que parece
Drive pero no está montado **no** es persistencia. La copia comprueba el hash del fichero visible; no demuestra que Google
haya terminado de sincronizarlo remotamente. No cierres la sesión inmediatamente después de copiar.

`files.download` inicia una transferencia hacia el navegador, no confirma que haya terminado. Puede empezar al finalizar
la celda y mantiene el fichero en RAM del navegador. El helper recomienda Drive y rechaza esa vía para ZIPs mayores de
512 MiB. No hay auto-clicks para evitar el timeout, túneles públicos ni tareas ocultas.


In [ ]:
from fencingly_v2 import colab_utils as CU
RUN_TRAINING = os.environ.get("FENCINGLY_RUN_TRAINING", "0") == "1"
USE_DRIVE = IN_COLAB
DRIVE_DIR = Path("/content/drive/MyDrive/FENCINGLY") if USE_DRIVE else None
if USE_DRIVE:
    CU.mount_drive()
    if not CU.drive_mounted():
        raise RuntimeError("Drive no está montado")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RUN_PARENT = Path(os.environ.get("FENCINGLY_RUN_PARENT", "/content/fencingly_runs" if IN_COLAB else str(ROOT.parent / "colab_runs")))
RUN_PARENT.mkdir(parents=True, exist_ok=True)
WORKSPACE_ZIP = os.environ.get("FENCINGLY_WORKSPACE_ZIP", "")
EXISTING_WORKSPACE = os.environ.get("FENCINGLY_WORKSPACE", "")
WORK = Path(EXISTING_WORKSPACE) if EXISTING_WORKSPACE else None
if WORKSPACE_ZIP:
    destination = RUN_PARENT / ("training_" + CW.stamp())
    inspection = AB.inspect_workspace(WORKSPACE_ZIP)
    print({key: inspection[key] for key in ("files", "total_bytes", "hashes_verified", "compatibility_checked")})
    restored = AB.restore_to_new_directory(WORKSPACE_ZIP, destination)
    WORK = Path(restored["destination"])
if RUN_TRAINING and (WORK is None or not (WORK / "store" / "store.json").is_file()):
    raise ValueError("Indica un workspace de 01 con Store válido; un ZIP de sólo evidencia no basta para entrenar")
EXPORT_DIR = RUN_PARENT / "exports"
EXPORT_DIR.mkdir(exist_ok=True)
print("WORK:", WORK)


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:preflight
# Mismas comprobaciones que en 01, mas el determinismo de la atencion. El orden importa:
# `determinismo()` va ANTES de construir el modelo.
import sys, os
from pathlib import Path
for _cand in (Path.cwd(), Path.cwd() / "paquete", Path("/content"), Path("/content/paquete")):
    if (_cand / "preflight.py").is_file() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))
import preflight as PF

PF.protocolo()
ENTORNO_PINES = PF.entorno(exigir=True)

# Backend de atencion `math`. Sin esto, cambiar SOLO el backend produce un "delta" de +0,36 pp con
# 327 de 3.047 decisiones distintas: un numero mayor que tres de los cinco efectos del ultimo
# barrido. Con el, max |p1-p2| = 0,000e+00 entre procesos distintos, sin coste de tiempo medible.
DETERMINISMO = PF.determinismo()

# La huella, otra vez: el interprete que entrena y el que materializo los artefactos tienen que ser
# el mismo, y esa es justo la propiedad que hace comparables las dos mitades del sistema.
HUELLA = PF.huella(exigir=True)

# Si este cuaderno construye algun HistGradientBoosting, tiene que fijar early_stopping: 'auto' se
# enciende por encima de 10.000 filas y la poblacion nueva son 12.930 filas con espejo, asi que YA
# cruza el umbral y las semillas empezarian a importar en silencio.
#   modelo = PF.estimador_hgb(max_iter=200, random_state=7)     # early_stopping=False explicito


## 1. Validar Store y selección sin tocar los splits guardados

El Store memmap no carga todos los arrays de una vez, pero el trainer puede cachear features/masks: la RAM sigue teniendo
un límite. `open_training_inputs` abre el Store con sus gates actuales, valida un índice elegible externo ligado por hash
y crea una selección fuera del Store. Su referencia `training/active_selection.json` se conserva y verifica al reanudar:
no se vuelve a generar una selección con otro hash sólo porque el registro restaurado cambió de ruta. No reescribe
`meta.csv`, `sample_meta.jsonl` ni arrays; no falsifica sus digests.

Puedes fijar `GLOBAL_REGISTRY` a tu registro existente. Si no lo haces, el notebook usa **sólo en lectura** el snapshot
que viajó con 01. Ese snapshot no se instala como autoridad de próximas descargas. También puedes reutilizar una selección
externa exacta con `SPLIT_SELECTION`; se valida contra este Store y sus identidades.

Se requieren las tres particiones y se comprueban cruces conocidos por clip, torneo, asalto, NPZ, vídeo y duplicados
registrados. El índice externo conserva claves que no caben en todas las columnas del Store. «Sin cruces conocidos» no
significa independencia humana o ausencia de replays no detectados. Un torneo/clip ya consultado para ajustar reglas no
se convierte en holdout nuevo por renombrarlo. Si falta test, se para; no se lo sustituye por val.

Un Store construido en otro runtime puede ser rechazado por su gate estricto aunque el workspace se haya transportado
bien. Conserva NPZ y el Store original; reconstruye una derivación nueva con el release adecuado si procede. No actives
legacy ni cambies hashes para eludirlo. La portabilidad de modelo v3 y la compatibilidad de Store son controles distintos.

### Sin preguntas: la forma del entrenamiento sale del workspace y del runtime

Este notebook ya no te pide épocas, lote ni dimensión. `training_shape` lee cuántas muestras trae el Store y
cuánta VRAM/RAM hay, y deriva `epochs`, `patience`, `d_model` y `batch_size`. Un Store pequeño recibe menos
épocas y menos capacidad; uno grande recibe más, siempre dentro de los límites conservadores de esta receta.
Las variables `FENCINGLY_EPOCHS`, `FENCINGLY_PATIENCE`, `FENCINGLY_D_MODEL` y `FENCINGLY_BATCH_SIZE` siguen
mandando si quieres fijar una de ellas. El plan derivado se imprime antes de entrenar y se guarda con la
trayectoria: reanudar mantiene el plan original, no lo recalcula con la RAM que tenga la VM nueva.


In [ ]:
GLOBAL_REGISTRY = os.environ.get("FENCINGLY_SPLIT_REGISTRY", "")
SPLIT_SELECTION = os.environ.get("FENCINGLY_SPLIT_SELECTION", "")
EXPORT_PORTABLE = True
USE_SAVED_TRAIN_CONFIG = True
EXPERIMENTAL_CUDA_RESUME = os.environ.get("FENCINGLY_EXPERIMENTAL_CUDA_RESUME", "0") == "1"
SIGNAL_EXPERIMENT = os.environ.get("FENCINGLY_SIGNAL_EXPERIMENT", "0") == "1"
VALIDATION_FOCUS = os.environ.get("FENCINGLY_VALIDATION_FOCUS", "two_valid_lights")
AUGMENTATION_PROFILE = os.environ.get("FENCINGLY_AUGMENTATION_PROFILE", "conservative")
SEARCH_AUGMENTATION = os.environ.get("FENCINGLY_SEARCH_AUGMENTATION", "1") == "1"
SEARCH_REGULARIZATION = os.environ.get("FENCINGLY_SEARCH_REGULARIZATION", "1") == "1"
TORCH_THREADS = 1
TRIAL_EPOCHS = 4
SEARCH_TIMEOUT_SECONDS = 900
DOWNLOAD_FINAL_IN_BROWSER = IN_COLAB
TRAINING_RESULT = None

if RUN_TRAINING:
    if not GLOBAL_REGISTRY:
        GLOBAL_REGISTRY = str(WORK / "provenance" / "split_registry_snapshot.json")
    if not SPLIT_SELECTION and not Path(GLOBAL_REGISTRY).is_file():
        raise ValueError("Falta registro GLOBAL existente o snapshot; no se reparten de nuevo los clips")
    ENVIRONMENT = CW.environment_report(WORK, stage="train", include_search=N_TRIALS > 0)
    if ENVIRONMENT["dependencies"]["missing"]:
        print("Instalando lo que falta para entrenar:", ENVIRONMENT["dependencies"]["missing"], flush=True)
        CW.install_missing("train", approved=True, include_search=N_TRIALS > 0)
        ENVIRONMENT = CW.environment_report(WORK, stage="train", include_search=N_TRIALS > 0)
    if ENVIRONMENT["dependencies"]["missing"]:
        raise RuntimeError(f"Faltan dependencias para entrenamiento/busqueda: {ENVIRONMENT['dependencies']['missing']}")
    PLAN = CW.resource_plan(ENVIRONMENT)
    AUTO = CW.training_shape(WORK, ENVIRONMENT, PLAN)
    print("Plan de entrenamiento derivado del workspace y del runtime:", AUTO)
    EPOCHS = int(os.environ.get("FENCINGLY_EPOCHS", AUTO["epochs"]))
    PATIENCE = int(os.environ.get("FENCINGLY_PATIENCE", AUTO["patience"]))
    D_MODEL = int(os.environ.get("FENCINGLY_D_MODEL", AUTO["d_model"]))
    BATCH_SIZE = int(os.environ.get("FENCINGLY_BATCH_SIZE", AUTO["batch_size"]))
    RAM_BUDGET = int(ENVIRONMENT["resources"]["ram"]["free_gb"] * .5 * 1024 ** 3)
    RECIPE_CLASS = RECIPE.SignalTrainingRecipe if SIGNAL_EXPERIMENT else RECIPE.TrainingRecipe
    SIGNAL_OPTIONS = dict(validation_focus=VALIDATION_FOCUS, augmentation_profile=AUGMENTATION_PROFILE,
                          search_augmentation=SEARCH_AUGMENTATION, search_regularization=SEARCH_REGULARIZATION) if SIGNAL_EXPERIMENT else {}
    TRAIN_CONFIG = RECIPE_CLASS(d=D_MODEL, T_max=135, epochs=EPOCHS, patience=PATIENCE,
        batch_size=min(BATCH_SIZE, PLAN["train_batch"]), trials=N_TRIALS, trial_epochs=TRIAL_EPOCHS,
        search_timeout_seconds=SEARCH_TIMEOUT_SECONDS, ram_budget_bytes=RAM_BUDGET, portable=EXPORT_PORTABLE,
        aux_weight=0.0, p_timewarp=0.0, weapon="foil", torch_threads=TORCH_THREADS,
        experimental_cuda_resume=EXPERIMENTAL_CUDA_RESUME, **SIGNAL_OPTIONS)
    TRAIN_CONFIG, CONFIG_ORIGIN = RECIPE.restored_recipe(WORK / "training", TRAIN_CONFIG, use_saved=USE_SAVED_TRAIN_CONFIG)
    print("Origen explicito de la configuracion:", CONFIG_ORIGIN)
    with RECIPE.open_training_inputs(WORK, split_registry=GLOBAL_REGISTRY, selection_path=SPLIT_SELECTION or None) as inputs:
        PREFLIGHT = RECIPE.validate_training_inputs(inputs["train"], inputs["val"], inputs["test"], TRAIN_CONFIG)
        if PREFLIGHT["memory"]["estimated_bytes"] > RAM_BUDGET:
            raise RuntimeError("La seleccion guardada excede la RAM libre actual: libera RAM o usa un nuevo experimento, no cambies su plan de forma silenciosa")
        SPLIT_SELECTION = inputs["selection_path"]
        print("Configuracion:", TRAIN_CONFIG)
        print("Validacion:", PREFLIGHT)
        print("Senales por particion:", PREFLIGHT["signals"])
        print("Avisos de cobertura de senales:", PREFLIGHT["signal_warnings"])
        print("Ambito para seleccionar el modelo:", PREFLIGHT["validation_focus"], PREFLIGHT["validation_samples_used_for_selection"])
        print("Seleccion externa:", SPLIT_SELECTION)
else:
    print("Modo seguro: no se abrio un Store ni se lanzo un optimizador")


## 2. Qué ve el modelo: T×F, máscaras y contexto post-tocado

Cada ejemplo es **un clip**, no cada frame como etiqueta independiente. Tras preparar a 30 fps y alinear por el final,
`X` tiene hasta T=135 posiciones y F=78 columnas actuales: 28 por tirador, cinco relacionales y 17 de contexto
(luces/tiempos/audio/disponibilidad). El orden exacto y la semántica proceden del contrato, nunca de reconstruir nombres
manualmente. `M` (T) indica posiciones utilizables por el modelo. Las masks por articulación/familia distinguen observado,
estimado con soporte corto e inválido; no se inventa un último frame válido cuando toda la muestra es missing.

Los ceros imputados **después** de normalizar son padding/input numérico, no mediciones de distancia, extensión o velocidad
cero. La web/informes usan la ruta de medidas que restaura missing/NaN y sus unidades; no dibujan esos ceros como datos reales.
El normalizador se ajusta con **train solamente**. Ni test, ni una tabla agregada de todas las armas entra en ese ajuste.

El endpoint es `post_touch_winner`: lado al que se concede el punto en el clip de florete. Puede ver la ventana posterior
al encendido, luces y audio. Por ello **no es anticipación**, predicción causal del futuro ni prueba de comprensión arbitral.
Recortar el array de features ya calculado no garantiza causalidad del prefijo. El buzzer puede seguir una luz/mezcla de
sonido; contacto de hoja y finta necesitan referencia independiente. Una doble de espada no es la misma etiqueta binaria.

Los rankings FIE, nombres y resultados sirven como contexto/procedencia para la web, no como un atajo oculto en X.
Las pseudoetiquetas LLM no son gold humano. `aux_weight=0` mantiene desactivadas las cabezas auxiliares experimentales.
Esta receta rechaza un valor positivo: esos ensayos necesitan aprobación/objetivos y evaluación independientes, no un
cambio silencioso para «mejorar» el número del notebook.

### Aumentación conservadora

Se aplica sobre pose, antes de recalcular features y masks. El espejo intercambia actores/luces e invierte left/right;
recorte inicial y pequeñas transformaciones de cámara mantienen desconocidos y fronteras. Jitter/dropout son leves.
Estas invariancias son hipótesis, no una garantía de prioridad. Se fija **p_timewarp=0**: el linaje de timewarp no está
certificado y podría enmascarar gran parte de los frames útiles. No se añaden segundos/píxeles interpolados ni se altera
independientemente la velocidad de un tirador para fabricar un orden de acción.

### Una luz, dos luces y sonido: no mezclar problemas

- **Una luz válida:** la inferencia identifica la señal detectada de ese lado, sujeta a anulación/revisión; no consulta al modelo para contradecirla con una hipótesis de movimiento. Una luz blanca no es una segunda luz válida. Una predicción o regla que favorezca a la izquierda no convierte una única señal válida derecha en punto izquierdo.
- **Dos luces:** se informa por separado rendimiento, denominador y abstenciones. La primera luz y la mayor velocidad no definen por sí solas la prioridad. El ganador de la fuente tampoco es automáticamente una anotación humana independiente de toda la frase.
- **Audio:** ya existen siete rasgos espectrales/transitorios y un indicador de disponibilidad. Pueden contener pisadas, público, hojas y zumbador; no son un detector independiente de contacto. Audio ausente, valores no utilizables y silencio medido son estados distintos.

El preflight imprime los conteos por partición y advierte si falta audio, casos de dos luces en test o ambos resultados
entre los casos difíciles de validación. Los metadatos explícitamente contradictorios se rechazan, sin cambiar `y` para
hacerlos coincidir con una hipótesis geométrica. Un registro desconocido no se convierte en cero luces.

### Comparación experimental centrada en los casos difíciles

`SIGNAL_EXPERIMENT=False` conserva la receta anterior. Para un **experimento nuevo** con datos apropiados, activa
`SIGNAL_EXPERIMENT=True`: `SignalTrainingRecipe` mantiene TemporalNet y F78, pero permite `VALIDATION_FOCUS="two_valid_lights"`.
El trainer recibe sólo ese subconjunto de **validación** para early stopping, poda y selección; el entrenamiento conserva
su selección original y los informes finales vuelven a separar todas las poblaciones. Si el subconjunto no existe o no
produce evaluaciones utilizables, se para; no se sustituye por casos fáciles ni por test. `VALIDATION_FOCUS="all"` es el control.

`AUGMENTATION_PROFILE` admite `none`, `flip` o `conservative`. Con `N_TRIALS>0`, los switches `SEARCH_AUGMENTATION` y
`SEARCH_REGULARIZATION` permiten buscar también perfil de aumentación, weight decay y label smoothing dentro de límites
pequeños. Más copias aumentadas no son más asaltos independientes. La receta elegida, el ámbito y los datos quedan registrados;
una receta guardada prevalece al reanudar. Para comparar otra, usa un workspace nuevo de 01, no sobrescribas una trayectoria.

No se han añadido columnas a pesos antiguos. Variables nuevas de mano armada, contacto, prioridad o audio separado necesitan
referencia, máscara, unidad y una representación versionada antes de presentarse como una mejora validada.

## 3. Entrenar con presupuesto y selección honesta

**Sin búsqueda:** N_TRIALS=0 utiliza la receta predeclarada; no imprime un «best» ficticio. **Con Optuna:** la receta base
busca d=48/64/96, lr/dropout conservadores, un solo trial a la vez, hasta ocho trials, ocho épocas de trial y 30 min como
límite de admisión. El timeout no cancela forzosamente una época ya en marcha. El objetivo es mínima `val_loss` finita
sobre el ámbito declarado: toda validación en la receta base, o sólo dos luces con `SIGNAL_EXPERIMENT=True` y el foco
correspondiente. Early stopping y poda reciben ese mismo subconjunto. Test no se usa para elegir arquitectura, umbral,
aumentación o época. El mínimo de un barrido pequeño no acredita un modelo mejor fuera de esos datos.

En la receta experimental, `SEARCH_REGULARIZATION=True` añade weight decay entre 0,0001 y 0,03 y label smoothing
0/0,05/0,1. `SEARCH_AUGMENTATION=True` compara `none`, `flip` y `conservative`, sin ampliar el número máximo de trials.
La configuración seleccionada conserva el ámbito de validación; cambiarlo exige otro experimento. No se selecciona por
una accuracy global que pueda ocultar mal rendimiento en dos luces, ni se presenta una prueba sintética como mejora real.

El trainer nativo utiliza AdamW, OneCycle, clipping, early stopping y masks. CUDA activa autocast FP16/GradScaler;
CPU permanece en FP32. Antes de empezar un fit CUDA nuevo se consulta VRAM y se prueba un lote con forward/backward
sintético de la misma forma T×78 y red pequeña, **sin pasos de optimizador**. Un OOM en esa prueba reduce el lote hasta uno.
Es una comprobación de capacidad, no precisión ni garantía contra todo OOM futuro. Una vez guardado el checkpoint inicial,
el lote efectivo y el plan OneCycle quedan fijados. Un OOM posterior exige liberar VRAM y reanudar con ese mismo lote;
**no se cambia bs para reiniciar silenciosamente desde cero**. Cambiar bs requiere otro experimento explícito.
El batch de los informes finales está acotado; su ruta portable es CPU. La evaluación interna por época sigue siendo
nativa. No se lanza un modelo grande en esta entrega ni se instala un stack GPU en una máquina CPU para simular una prueba.

### Reanudación de verdad en frontera de época

El fit final crea `training/epochs/epoch_XXXXXX_<id>.epoch.zip`: primero la frontera cero, después cada época completa.
Cada archivo es NUEVO, validado y publicado atómicamente en disco local; no reemplaza ni borra checkpoints anteriores.
El ZIP contiene sólo `checkpoint.json` y `tensors.bin`, usando el codec numérico acotado del proyecto, **sin pickle ni
objetos Python suministrados por el archivo**. Es distinto del bundle de inferencia v3; no se carga como modelo de la web.

| Estado persistido | Por qué hace falta |
|---|---|
| Pesos actuales y mejores, por separado | El optimizador continúa desde los actuales, no desde otra época seleccionada como best |
| AdamW: momentos, pasos y grupos | Mantiene la trayectoria del optimizador, no sólo una inicialización de pesos |
| OneCycle: plan total y estado | Mantiene la misma LR/momentum; no se recalcula con las épocas restantes |
| GradScaler | Mantiene el estado AMP; pasos omitidos por overflow no se cuentan como actualizaciones |
| Época siguiente, historia, mejor época/pérdida, paciencia y contadores | Respeta early stopping y terminales; un modelo ya terminado no se sigue ajustando |
| RNG Torch CPU/CUDA, NumPy global, SeqData train/val y Python | Conserva orden y aumentaciones bajo el entorno admitido |
| Normalizador, configuración, contrato, digest completo de datos/selección y runtime | Impide continuar con otro dataset, máscaras, etiquetas, plan o código |

Se exige **num_workers=0** para estos checkpoints. No se afirma recuperar RNG de workers persistentes que no se ha guardado:
con workers >0 se rechaza el modo de reanudación. Una época cortada a mitad se vuelve a ejecutar desde su frontera anterior;
no hay reanudación exacta de minibatch. Si la frontera inicial todavía no se había podido guardar, el helper no inventa un
checkpoint ni llama warm start a una continuación exacta. Un estado corrupto o incompatible se rechaza, sin buscar otro
archivo y ocultar el error.

**No pongas EPOCHS=K para simular un corte.** Define desde el inicio el plan completo N. Al volver, mantén N, bs efectivo,
T_max, dropout/lr/augment, datos, máscaras, selección y versiones/runtime. `USE_SAVED_TRAIN_CONFIG=True` selecciona
explícitamente la configuración guardada y la imprime: evita cambiarla porque otra VM tenga distinta RAM libre. Los knobs
editables de configuración nueva sólo mandan si desactivas esa opción; entonces cualquier diferencia con el checkpoint
se rechaza. La memoria libre actual se comprueba igualmente. La selección original se conserva mediante su referencia
externa; no se modifica el Store ni se recrea un split desde el inventario maestro de análisis.

La equivalencia numérica se prueba en **CPU, mismo entorno**: ejecución N épocas frente a corte tras K y reanudación,
comparando pesos, pérdidas/historia sin tiempos, orden, momentos, scheduler y RNG. Eso no es validación deportiva.
La rama CUDA guarda el estado correspondiente, pero no se ha ejecutado aquí una GPU real. Por ello reanudar CUDA requiere
`EXPERIMENTAL_CUDA_RESUME=True` consciente; por defecto aparece un error claro. No se garantiza identidad bit a bit de
kernels CUDA ni continuidad entre T4/L4/drivers/versiones distintos. El cambio de hardware/runtime se detecta y se rechaza
como reanudación exacta. No se amplían tolerancias ni se relajan gates para hacerlo pasar.

### Qué sobrevive a la VM y qué no

El callback copia snapshots completos a Drive después de los commits de época cuando Drive está habilitado. El checkpoint
local por sí solo no sobrevive a la destrucción de `/content`. Restaura el último ZIP confirmado en una carpeta NUEVA,
abre el mismo release y vuelve a ejecutar 02: se usa la última frontera comprometida sin reajustar el normalizador ni
repetir las épocas terminadas. Si sólo falló el informe, se reutiliza el modelo ya guardado. Los hashes son integridad,
no autenticación del autor ni demostración de independencia de la muestra.

Optuna conserva trials terminados/podados/fallidos en JSON sin exigir SQLite. La continuación de optimizador descrita
arriba corresponde al **fit final**; no se promete estado exacto del muestreador/podador de Optuna ni continuar un trial
interrumpido a mitad. Una búsqueda incompleta y un fit final interrumpido son estados diferentes. N_TRIALS=0 es la ruta
conservadora predeterminada; nunca devuelve parámetros que activen timewarp o auxiliares no admitidos.

Los snapshots completos consumen disco/transferencia y no se borran automáticamente. El modelo final queda una sola vez
en `WORK/model_bundle.zip`; para otro experimento usa otro WORK. El borrado de MP4 sigue limitado a descargas propias de 01
con sus commits válidos: los checkpoints de entrenamiento, modelos, historia y originales no se borran.

In [ ]:
if RUN_TRAINING:
    def snapshot_training(directory, phase):
        model_path = directory.parent / "model_bundle.zip"
        return CW.publish_workspace(directory.parent, EXPORT_DIR,
            model_bundle=model_path if model_path.is_file() else None,
            split_registry=GLOBAL_REGISTRY, drive_dir=DRIVE_DIR,
            metadata={"phase": phase, "model_validation": "see_training_state", "independence": "not_established"})
    with RECIPE.open_training_inputs(WORK, selection_path=SPLIT_SELECTION) as inputs:
        TRAINING_RESULT = RECIPE.run_training(inputs["train"], inputs["val"], inputs["test"], WORK / "training",
            recipe=TRAIN_CONFIG, device=PLAN["device"], on_checkpoint=snapshot_training if DRIVE_DIR is not None else None)
    MODEL_BUNDLE = Path(TRAINING_RESULT["model_bundle"])
    print("Estado:", TRAINING_RESULT["state"]["state"])
    print("Bundle:", MODEL_BUNDLE)
    print("Reutilizado sin entrenar:", TRAINING_RESULT["resumed_completed_phase"])
else:
    print("No hay entrenamiento, Optuna ni pesos nuevos en este modo de revisión")


## 4. Leer los resultados sin convertir cobertura en acierto

Se generan informes separados **train / val / test**, desgloses por torneo/asalto y una/dos luces. Incluyen n total,
n scored, abstenciones, cobertura, accuracy, balanced accuracy cuando ambas clases están presentes, macro-F1, matriz de
confusión, NLL, Brier, ECE y riesgo-cobertura. Un null con cero denominador no es un error deportivo de cero; una tabla con
n=2 tampoco soporta conclusiones estadísticamente precisas. No se publican aquí ganancias ficticias ni intervalos por frame
que traten ventanas correlacionadas como observaciones independientes.

Baselines: mayoría calculada **sólo en train**, una luz realmente observada cuando está disponible y primera luz como
heurística explícita, **no regla de prioridad**. Un empate de frame de luces no asigna ganador. Cada baseline tiene su
propia cobertura y además una comparación pareada con el modelo sobre el mismo subconjunto; no se comparan porcentajes de
universos distintos como si fuesen ganancia. Un baseline ausente permanece ausente.

ECE es una medición de calibración sobre estas etiquetas, no una certificación de confianza. La receta conserva
`temperature=1` salvo un protocolo separado y explícito; no ajusta temperatura/umbral con test. Los umbrales mostrados son
una curva descriptiva, no una recomendación ya validada de despliegue. Antes de alegar mejora arbitral hacen falta humanos
independientes, blinding, nuevos grupos, error temporal y un holdout no gastado.

Para portable=True, los informes finales se calculan con el bundle **recargado en CPU FP32**, el perfil de conformidad
portable admitido. El fit pudo usar CUDA, pero ello no convierte la inferencia portable en certificación GPU. Se conserva
por separado el runtime de entrenamiento y el receipt de verificación local.


In [ ]:
if TRAINING_RESULT is not None:
    import pandas as pd
    EVALUATION = TRAINING_RESULT["evaluation"]
    display(pd.DataFrame({split: report["overall"] for split, report in EVALUATION.items()}).T[
        ["n", "n_scored", "n_abstained", "coverage", "accuracy", "balanced_accuracy", "macro_f1", "nll", "brier", "ece"]])
    print("Dos luces y una luz:")
    display(EVALUATION["test"]["groups"]["n_valid_lights"])
    print("Baselines con denominadores y comparación pareada:")
    display(EVALUATION["test"]["paired_comparisons"])
    print("Receipt portable local:", getattr(TRAINING_RESULT["model"], "portable_verification", None))


## 5. Bundle portable + workspace final: llevarlo a la web

`EXPORT_PORTABLE=True` llama explícitamente a `train.save_bundle(..., portable=True)`. Si la firma no existe en tu release,
se para: no se activa legacy, no se edita un fingerprint y no se degrada silenciosamente a otro formato. El formato v3
lleva `weights.bin` numérico acotado, `meta.json`, `attestation.json`, `conformance.json` y normalizador si corresponde.
No necesita ejecutar pickle/código aportado por un usuario. El formato estricto v2 sigue siendo otro contrato, no se
rebautiza un archivo legacy como v3.

La recarga comprueba esquema, arquitectura, orden/semántica de columnas, normalizador, fuentes/bindings y vectores de
conformidad. Conserva el contrato/runtime de entrenamiento original y expone una verificación local aparte. Los pesos
portables recargados **no se reemiten ni reetiquetan**: se transporta su ZIP original byte a byte. El SHA prueba integridad,
no quién generó las etiquetas. Los vectores finitos y las simulaciones de versiones no son una garantía universal
Windows↔Colab↔T4: hay que ejecutar el artefacto concreto en ambos runtimes y conservar el resultado.

La salida fundamental es el **workspace completo**, no sólo el modelo: la web necesita manifiesto, NPZ, movimientos,
QC, Store cuando proceda y caché FIE para disponer de la evidencia. El transporte jamás ejecuta el contenido del ZIP y
restaura en carpeta nueva. Luego el servidor de confianza valida/carga el bundle por separado. Un rechazo de modelo no
justifica borrar datos ni ocultar el error; conserva el ZIP original y revisa la compatibilidad de la misma entrega.


In [ ]:
FINAL_EXPORT = None
if TRAINING_RESULT is not None:
    FINAL_EXPORT = CW.publish_workspace(WORK, EXPORT_DIR, model_bundle=MODEL_BUNDLE, split_registry=GLOBAL_REGISTRY,
        drive_dir=DRIVE_DIR, browser_download=DOWNLOAD_FINAL_IN_BROWSER,
        metadata={"phase": "trained_workspace", "training_state": TRAINING_RESULT["state"]["state"],
                  "label_scope": "foil_post_touch_source_labels", "independent_accuracy_claim": False, "release": RELEASE_PROVENANCE})
    print("IMPORTA ESTE ZIP EN LA WEB LOCAL:", FINAL_EXPORT.get("drive_path", FINAL_EXPORT["path"]))
    print("SHA-256:", FINAL_EXPORT["sha256"])
    print("Modelo original (también dentro del workspace):", MODEL_BUNDLE)
else:
    print("No se exportó un modelo sin entrenar/validar")


## Ya has terminado los dos pasos efectivos

Abre la web local incluida en `FENCINGLY/webapp`, importa el workspace final por su ruta de importación segura y elige los
MP4 originales sólo si quieres reproducir vídeo. Los gráficos de NPZ/QC/pose pueden funcionar sin esos originales; no se
incluyen por defecto por tamaño y privacidad. La integración exacta de importación y la validación de navegador pertenecen
a la aplicación de la entrega; este notebook no promete un frontend alternativo.

Si no puedes o no quieres iniciar la web en tu PC, abre **03_Shared_Web_App.ipynb**, opcional. Servirá el mismo
`webapp_react/server.py` y su mismo `frontend/dist`, no una copia de interfaz. Si falla un gate, conserva el error y los
artefactos. No uses `allow_legacy=True` como receta de producción. No publiques el release, los datos ni el modelo en un
repositorio para hacer funcionar estos notebooks privados.


In [ ]:
# @@FENCINGLY_DURACION_COMPLETA:dos_poblaciones
# LOS TRES BRAZOS DE ELIMINACION DEL ATAJO, con el signo declarado ANTES de medir.
#
# Declararlo antes es lo que evita leer un resultado al reves. Cada brazo imprime que hace, su base
# medida, que se sabe ya de el, el signo esperado y COMO SE LEERIA AL REVES.
for _b in ("A_ventana_simetrica", "B_reponderar_960", "C_invertir_luces"):
    PF.declarar_brazo(_b)
    print()

# El veredicto, en las dos poblaciones. Rellena las tres series con las de tu evaluacion:
#   y_verdad         : etiqueta real por clip (0/1)
#   y_pred           : prediccion del artefacto. PROMEDIA PROBABILIDADES sobre semillas y umbraliza
#                      UNA vez. NO promedies el acierto por clip: no es la misma cantidad, y con el
#                      convenio del artefacto un efecto de +1,03 fue +0,80 CON EL INTERVALO
#                      CRUZANDO EL CERO. Usa el del artefacto y solo el del artefacto.
#   gano_el_segundo  : True si en ese clip gano el que encendio SEGUNDO. Es una propiedad del
#                      RESULTADO, asi que solo DESCRIBE donde cayo la ganancia: nunca es la
#                      poblacion de evaluacion, ni objetivo de optimizacion, ni puerta de enrutado.
#
# VEREDICTO = PF.dos_poblaciones(y_verdad, y_pred, gano_el_segundo)
#
# Y el resultado NO se resta del 73,88 % ni del 70,04 %: es una linea base nueva, por la ventana Y
# por la huella (el recorte de export_measurements cambio producer_sha256 a proposito).
print("Rellena las tres series y llama a PF.dos_poblaciones. Ver la celda de PROTOCOLO.")
